# វិភាគការទាមទារចំណាយ

កំណត់សម្គាល់នេះបង្ហាញពីវិធីបង្កើត agents ដែលប្រើ plugins ដើម្បីដំណើរការចំណាយសម្រាប់ការធ្វើដំណើរពីរូបភាពវិក័យប័ត្រផ្ទាល់ (local receipt images), បង្កើតអ៊ីមែលទាមទារចំណាយ និងបង្ហាញទិន្នន័យចំណាយជាក្រាបរង្វង់ (pie chart). Agents ជ្រើសរើសមុខងារយ៉ាងឆ្លាតវៃ ដោយផ្អែកលើបរិបទភារកិច្ច។

Steps:
1. OCR Agent processes the local receipt image and extracts travel expense data.
2. Email Agent generates an expense claim email.

### ឧទាហរណ៍ស្ថានភាពចំណាយធ្វើដំណើរ:
សូមសន្មតថាអ្នកជាវិក្តុបនិយោជិតដែលធ្វើដំណើរសម្រាប់ការប្រជុំអាជីវកម្មនៅក្នុងទីក្រុងផ្សេង។ ក្រុមហ៊ុនរបស់អ្នកមានគោលនយោបាយសងប្រាក់ត្រឡប់ចំពោះចំណាយដែលសមស្របទាក់ទងនឹងការធ្វើដំណើរ។ ខាងក្រោមជាការបំបែកចំណាយធ្វើដំណើរដែលអាចកើតមាន៖
- ការដឹកជញ្ជូន:
សំបុត្រយន្តហោះសម្រាប់ដំណើរម្ដងចូលម្តងចេញពីទីក្រុងភូមិរបស់អ្នកទៅទីក្រុងគោលដៅ។
សេវាតាក់ស៊ី ឬសេវាហៅឡានទៅនិងពីព្រលានយន្តហោះ។
ការដឹកជញ្ជូនក្នុងទីក្រុងគោលដៅ (ដូចជា បណ្តាញចរាចរណ៍សាធារណៈ ឡានជួល ឬតាក់ស៊ី)។

- ការស្នាក់នៅ:
ការស្នាក់នៅសណ្ឋាគារពីរបីយប់នៅសណ្ឋាគារអាជីវកម្មថ្នាក់មធ្យម ដែលនៅជិតកន្លែងប្រជុំ។

- អាហារ:
ប្រាក់ឧបត្ថម្ភអាហារប្រចាំថ្ងៃសម្រាប់អាហារពេលព្រឹក ថ្ងៃត្រង់ និងល្ងាច ដោយយោងទៅតាមគោលនយោបាយប្រាក់ប្រចាំថ្ងៃរបស់ក្រុមហ៊ុន។

- ចំណាយផ្សេងៗ:
កម្រៃចតយានយន្តនៅព្រលានយន្តហោះ។
កម្រៃចូលប្រើអ៊ីនធឺណិតនៅសណ្ឋាគារ។
ទឹកប្រាក់ជូន (tips) ឬកម្រៃសេវាកម្មតូចៗ។

- ឯកសារ:
អ្នកដាក់ស្នើវិក័យប័ត្រទាំងអស់ (សំបុត្រយន្តហោះ តាក់ស៊ី សណ្ឋាគារ អាហារ ល។) និងរបាយការណ៍ចំណាយដែលបានបំពេញសម្រាប់ការសងប្រាក់ត្រឡប់។


## Import required libraries

នាំចូលបណ្ណាល័យ និងម៉ូឌុលដែលចាំបាច់សម្រាប់កំណត់ត្រា។


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## កំណត់ម៉ូឌែលចំណាយ

 បង្កើតម៉ូឌែល Pydantic សម្រាប់ចំណាយមួយៗ និងថ្នាក់ ExpenseFormatter ដើម្បីបម្លែងសំណើរបស់អ្នកប្រើទៅជាទិន្នន័យចំណាយដែលមានរចនាសម្ព័ន្ធ។

 គ្រប់ចំណាយនឹងត្រូវតំណាងដោយទ្រង់ទ្រាយ:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## ការកំណត់ឧបករណ៍ - ការបង្កើតអ៊ីមែល

បង្កើតមុខងារ ឧបករណ៍ មួយ ដើម្បីបង្កើតអ៊ីមែលសម្រាប់ដាក់ស្នើសំណើសុំចំណាយ.
- ឧបករណ៍នេះប្រើ `@tool` decorator ពី Microsoft Agent Framework.
- វាគណនាចំនួនសរុបនៃចំណាយ ហើយរៀបចំលម្អិតទៅក្នុងខ្លឹមសារ​អ៊ីមែល.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ឧបករណ៍សម្រាប់ដកចំណាយការធ្វើដំណើរពីរូបភាពវិក្កយបត្រ

បង្កើតមុខងារមួយសម្រាប់ឧបករណ៍ ដើម្បីដកចំណាយសម្រាប់ការធ្វើដំណើរពីរូបភាពវិក្កយបត្រ.
- ឧបករណ៍នេះប្រើ `@tool` decorator ពី Microsoft Agent Framework.
- វាអានរូបភាពវិក្កយបត្រ, កូដវាទៅជា base64, ហើយត្រឡប់ជា data URI សម្រាប់ឱ្យភ្នាក់ងារវិភាគ.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ការដំណើរការ​ចំណាយ

កំណត់ភ្នាក់ងារ និងភ្ជាប់ពួកវាទៅក្នុងលំហូរដំណើរការតាមលំដាប់ ដោយប្រើ `WorkflowBuilder`.
- ភ្នាក់ងារ OCR ដកយកទិន្នន័យចំណាយដែលមានរចនាសម្ព័ន្ធពីរូបភាពបង្កាន់ដៃ ដោយប្រើឧបករណ៍ `load_receipt_image`.
- ភ្នាក់ងារ Email ទទួលយកទិន្នន័យដែលបានដកស្រង់ ហើយបង្កើតអ៊ីមែលសំណើសុំសងចំណាយដែលមានវិជ្ជាជីវៈ ដោយប្រើឧបករណ៍ `generate_expense_email`.
- `WorkflowBuilder` ជាមួយ `add_edge` បង្កើតបណ្តាញបន្ទាត់តាមលំដាប់: ភ្នាក់ងារ OCR → ភ្នាក់ងារ Email.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## មុខងារ​ចម្បង

សាងសង់ដំណើរការ​តាមលំដាប់ និងដំណើរការវា ដើម្បីដំណើរការរូបភាពបង្កាន់ដៃ និងបង្កើតអ៊ីមែលសំណើរសុំសងចំណាយ។


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែដោយបញ្ញាសិប្បនិម្មិត (AI) [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះបីយើងខិតខំរកភាពត្រឹមត្រូវក៏ដោយ សូមប្រយ័ត្នកូវថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកថាជាប្រភពផ្លូវការដែលអាចយោងបាន។ សម្រាប់ព័ត៌មានសំខាន់ យើងផ្តល់អនុសាសន៍ឲ្យប្រើការបកប្រែមនុស្សដែលមានវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកអត្ថន័យខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
